# Data generation - Cellmates model

In [1]:
import networkx as nx
import numpy as np
import scipy.stats as ss
import dendropy
import math
import random
from simulation.datagen import rand_dataset


In [2]:
n_cells = 5
n_states = 10
K = n_states - 1
n_sites = 200

In [3]:
from simulation.datagen import get_ctr_table

data = rand_dataset(n_cells, n_states, n_sites, alpha=1., obs_type='pois', p_change=0.2, seed=1)
tree = data['tree']
ctr = get_ctr_table(tree)

In [4]:
data['tree'].print_plot(plot_metric='length')

                                                     /c1                       
                                                     |                         
               /-------------------------------------+              /-- c0     
               |                                     |         /----+          
               |                                     \---------+    \----- c2  
               +                                               |               
               |                                               \------------ c3
               |                                                               
               \-------------- c4                                              
                                                                               
                                                                               


In [5]:
print(ctr)
for node in data['tree'].preorder_node_iter():
    print(f"Node {node.label} edge length: {node.edge_length}")
# tree to newick
print(data['tree'].as_string('newick'))


[[0.         0.03202061 0.04505685 0.04106825 0.        ]
 [0.         0.         0.03202061 0.03202061 0.        ]
 [0.         0.         0.         0.04106825 0.        ]
 [0.         0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.        ]]
Node 8 edge length: 0.013561073248783183
Node 7 edge length: 0.032020605951930334
Node 1 edge length: 2.874568575338041e-06
Node 6 edge length: 0.009047639965994664
Node 5 edge length: 0.003988601117904351
Node 0 edge length: 0.0024348314814298705
Node 2 edge length: 0.005179958142466622
Node 3 edge length: 0.010655140712989498
Node 4 edge length: 0.012702751654316759
[&R] ((c1:2.874568575338041e-06,((c0:0.0024348314814298705,c2:0.005179958142466622)5:0.003988601117904351,c3:0.010655140712989498)6:0.009047639965994664)7:0.032020605951930334,c4:0.012702751654316759)8:0.013561073248783183;



In [6]:
len(tree.nodes())

9

In [7]:
def pdd(l, n_states, alpha=1.):
    return (1 + (n_states - 1) * math.exp(-n_states * alpha * l)) / n_states

for edge in tree.postorder_edge_iter():
    print(edge.length, pdd(edge.length, n_states, alpha=.05))



2.874568575338041e-06 0.9999987064450707
0.0024348314814298705 0.9989049925082775
0.005179958142466622 0.9976720348277581
0.003988601117904351 0.9982069180633897
0.010655140712989498 0.9952179363801198
0.009047639965994664 0.9959377573702903
0.032020605951930334 0.9857054626006537
0.012702751654316759 0.9943018763730571
0.013561073248783183 0.9939181594108538


In [8]:
from simulation.datagen import simulate_cn_seq

In [9]:
cn = np.empty((len(tree.nodes()), n_sites))
cn[0, :] = 2
for i, n in enumerate(tree.preorder_node_iter()):
    n.label = i
    if i != 0:
        cn[i] = simulate_cn_seq(cn[n.parent_node.label], n_states, n.edge_length, alpha=.01)


In [10]:
cn[:10, :20]

array([[2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.],
       [2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2., 2.,
        2., 2., 2., 2.]])

In [11]:
print(tree.taxon_namespace)
for i, t in enumerate(tree.leaf_node_iter()):
    print(t.label, t.taxon.label)

['c0', 'c1', 'c2', 'c3', 'c4']
2 c1
5 c0
6 c2
7 c3
8 c4


In [12]:
import networkx as nx

newick = tree.as_string('newick')
dendropy_tree = dendropy.Tree.get(data=newick, schema='newick', taxon_namespace=tree.taxon_namespace)
print(dendropy_tree.seed_node)

<Node object at 0x10e5d33a0: 'None' (None)>


In [13]:
# Create an empty directed graph in NetworkX
G = nx.DiGraph()

# Function to recursively add nodes and edges to the NetworkX graph
def add_edges(G, node):
    for child in node.child_node_iter():
        # Add an edge between the parent and the child
        G.add_edge(node.label, child.label)
        # Recursively add the children of the current node
        add_edges(G, child)

# Start from the root node and add edges to the graph
add_edges(G,tree.seed_node)
nx.write_network_text(G)
assert nx.is_arborescence(G)

╙── 0
    ├─╼ 1
    │   ├─╼ 2
    │   └─╼ 3
    │       ├─╼ 4
    │       │   ├─╼ 5
    │       │   └─╼ 6
    │       └─╼ 7
    └─╼ 8


In [14]:
from utils.tree_utils import convert_networkx_to_dendropy
dtree = convert_networkx_to_dendropy(G, labels_mapping={l.label: l.taxon.label for l in tree.leaf_node_iter()},
                                     taxon_namespace=tree.taxon_namespace)
dtree.print_plot()

                   /-------------------------------------------------------- c1
                   |                                                           
/------------------+                                     /------------------ c0
|                  |                  /------------------+                     
|                  \------------------+                  \------------------ c2
+                                     |                                        
|                                     \------------------------------------- c3
|                                                                              
\--------------------------------------------------------------------------- c4
                                                                               
                                                                               


In [19]:
from dendropy.calculate.treecompare import robinson_foulds_distance, symmetric_difference

print(symmetric_difference(tree.))

0


ValueError: invalid literal for int() with base 10: 'c1'

In [ ]:

dendropy_tree.print_plot(plot_metric='length')
